# **Processing MedQA dataset**

## **MedQA Test Set formatting**
This script converts the raw MedQA test dataset (`phrases_no_exclude_test.jsonl`) into a standardized format for model evaluation. It structures each entry with a System Instruction, the Question and Options as the input, and isolates the Correct Letter (A, B, C, or D) as the output.

In [ ]:
import json
import os

# --- CONFIGURATION ---
INPUT_FILE = "../../../Data/MedQA/US/Data/4_options/phrases_no_exclude_test.jsonl"
OUTPUT_FILE = "medqa_test.jsonl"

SYSTEM_INSTRUCTION = (
    "You are a clinical decision support assistant taking a medical certification exam. "
    "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) "
    "corresponding to the correct answer."
)

def process_medqa_test_data():
    print(f" Starting conversion of {INPUT_FILE}...")
    
    if not os.path.exists(INPUT_FILE):
        print(f" Error: Input file '{INPUT_FILE}' not found.")
        return

    processed_count = 0
    count = 0
    
    with open(INPUT_FILE, 'r', encoding='utf-8') as fin, \
         open(OUTPUT_FILE, 'w', encoding='utf-8') as fout:
        
        for line in fin:
            try:
                item = json.loads(line)
                
                question = item.get('question', '').strip()
                options_dict = item.get('options', {})
                correct_letter = item.get('answer_idx', '').strip()
                
                if not question or not options_dict or not correct_letter:
                    count +=1
                    continue

                # Format Options (A. Text \n B. Text ...)
                # Sorted to ensure A, B, C, D order
                formatted_options = []
                for key in sorted(options_dict.keys()):
                    formatted_options.append(f"{key}. {options_dict[key]}")
                
                options_string = "\n".join(formatted_options)
                
                # formatting: "###Question: Text\n###Options:\nA. Option..."
                final_input = (
                    f"###Question: {question}\n"
                    f"###Options:\n{options_string}"
                )

                eval_entry = {
                    "instruction": SYSTEM_INSTRUCTION,
                    "input": final_input,
                    "output": correct_letter  
                }
                
                fout.write(json.dumps(eval_entry, ensure_ascii=False) + '\n')
                processed_count += 1
                
            except json.JSONDecodeError:
                continue

    print(f"Completed")
    print(f"   - Total questions processed: {processed_count}")
    print(f"   - Total number of lines without the complete information: {count}")
    print(f"   - Saved to: {OUTPUT_FILE}")

process_medqa_test_data()

 Starting conversion of ../../../Data/MedQA/US/Data/4_options/phrases_no_exclude_test.jsonl...
Completed
   - Total questions processed: 1273
   - Total number of lines without the complete information: 0
   - Saved to: medqa_test.jsonl


## **Few shot prompting**

Uses entries from the train dataset

In [ ]:
import json
import os
import random

random.seed(42)

# --- CONFIGURATION ---
INPUT_FILE = "../../../Data/MedQA/US/Data/4_options/phrases_no_exclude_train.jsonl"
OUTPUT_FILE = "few_shot_examples_medqa.jsonl"

SYSTEM_INSTRUCTION = (
    "You are a clinical decision support assistant taking a medical certification exam. "
    "Read the question and options carefully. Output ONLY the letter (A, B, C, or D) "
    "corresponding to the correct answer."
)

def generate_medqa_fewshot():
    print(f"--- [START] Generating 5-Shot Examples for MedQA (TRAIN DATA) ---")
    
    if not os.path.exists(INPUT_FILE):
        print(f" [ERROR] Input file '{INPUT_FILE}' not found.")
        return

    all_train_questions = []
    skipped_count = 0

    # 1. Read and format all training questions
    with open(INPUT_FILE, 'r', encoding='utf-8') as fin:
        for line in fin:
            try:
                item = json.loads(line)
            
                question = item.get('question', '').strip()
                options_dict = item.get('options', {})
                correct_letter = item.get('answer_idx', '').strip()
                
                if not question or not options_dict or not correct_letter:
                    skipped_count += 1
                    continue

                # Format Options (Sorted to ensure A, B, C, D order)
                formatted_options = []
                for key in sorted(options_dict.keys()):
                    formatted_options.append(f"{key}. {options_dict[key]}")
                
                options_string = "\n".join(formatted_options)
                
                final_input = (
                    f"###Question: {question}\n"
                    f"###Options:\n{options_string}"
                )

                eval_entry = {
                    "instruction": SYSTEM_INSTRUCTION,
                    "input": final_input,
                    "output": correct_letter 
                }
                
                all_train_questions.append(eval_entry)
                
            except json.JSONDecodeError:
                continue

    print(f" [INFO] Successfully parsed {len(all_train_questions)} valid questions from the training set.")

    # Select exactly 5 examples for Few-Shot
    if len(all_train_questions) < 5:
        print(" [ERROR] Not enough questions to generate a 5-shot prompt.")
        return

    # random.sample guarantees 5 unique questions without repetition
    final_few_shot = random.sample(all_train_questions, 5)

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as fout:
        for item in final_few_shot:
            fout.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f" -> Success! Generated 5 MedQA Few-Shot Examples.")
    print(f" -> Incomplete lines skipped: {skipped_count}")
    print(f" -> Saved to: {OUTPUT_FILE}\n")

if __name__ == "__main__":
    generate_medqa_fewshot()

--- [START] Generating 5-Shot Examples for MedQA (TRAIN DATA) ---
 [INFO] Successfully parsed 10178 valid questions from the training set.
 -> Success! Generated 5 MedQA Few-Shot Examples.
 -> Incomplete lines skipped: 0
 -> Saved to: few_shot_examples_medqa.jsonl

